|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Tensor parallelism<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: two shardings, one collective<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import torch
torch.manual_seed(0)
HIDDEN, MLP_HIDDEN, RANKS = 256, 1024, 4
inputs = torch.randn(8, HIDDEN)
up_weight = torch.randn(HIDDEN, MLP_HIDDEN) / HIDDEN**0.5
down_weight = torch.randn(MLP_HIDDEN, HIDDEN) / MLP_HIDDEN**0.5
reference = torch.relu(inputs @ up_weight) @ down_weight
print('reference', tuple(reference.shape))

Shard an MLP across four ranks in two different ways. Both ways are correct.
Count what each one costs in conversation.

All of it runs on the CPU. Stage 20 does the same work with real collectives.
This notebook is about the design rule. You must get the rule correct before
the collectives exist.

# Exercise 1: column, then row

In [ ]:
def mlp_parallel(inputs, up_weight, down_weight, ranks):
  """-> (output, number of all-reduces). Column-parallel, then row-parallel."""
  up_shards = list(up_weight.chunk(ranks, dim=1))      # COLUMNS
  down_shards = list(down_weight.chunk(ranks, dim=0))  # ROWS, to match
  partials = [torch.relu(inputs @ up_shards[rank]) @ down_shards[rank]
              for rank in range(ranks)]
  return sum(partials), 1                              # one all-reduce

output, collectives = mlp_parallel(inputs, up_weight, down_weight, RANKS)
print(f'max difference {(output-reference).abs().max().item():.2e}')
print(f'collectives for each block: {collectives}')

# Exercise 2: the other way round

Also correct. Count the collectives.

In [ ]:
def mlp_row_first(inputs, up_weight, down_weight, ranks):
  """Row-parallel first. The ranks must now gather the hidden activations
  BEFORE relu, because each rank holds a partial sum of them."""
  up_shards = list(up_weight.chunk(ranks, dim=0))       # rows: divides the INPUT
  input_shards = list(inputs.chunk(ranks, dim=1))
  hidden = sum(input_shards[rank] @ up_shards[rank] for rank in range(ranks))   # collective 1
  hidden = torch.relu(hidden)
  down_shards = list(down_weight.chunk(ranks, dim=0))
  hidden_shards = list(hidden.chunk(ranks, dim=1))
  output = sum(hidden_shards[rank] @ down_shards[rank] for rank in range(ranks))  # collective 2
  return output, 2

row_first_output, row_first_collectives = mlp_row_first(inputs, up_weight, down_weight, RANKS)
print(f'max difference {(row_first_output-reference).abs().max().item():.2e}   (still correct)')
print(f'collectives for each block: {row_first_collectives}   <- twice the talking, same answer')

# Exercise 3: what a collective costs, with no clock

Each side of this comparison is bytes divided by a bandwidth, so the
milliseconds cancel. Write the ratio of collective time to compute time. You
then find that the layer count also cancels.

Three things remain: your model, your batch, and one number about the machine.
That number is how many times faster the memory is than the interconnect.

In [ ]:
def comm_over_compute(batch, hidden, ranks, bandwidth_ratio, collectives=1):
  """Collective time divided by compute time. It has no unit and no clock.

       comm      B C (R-1)     HBM
       ----  =  -----------  x -----
       step        6 d           I
  """
  return batch * collectives * (ranks-1) / (6*hidden) * bandwidth_ratio

def break_even_batch(hidden, ranks, bandwidth_ratio, collectives=1):
  """The batch at which the ratio is 1: the ranks talk as long as they compute."""
  return 6*hidden / ((ranks-1) * bandwidth_ratio * collectives)

BANDWIDTH_RATIO = 25.0     # a PCIe node: HBM is approximately 25x the interconnect
print(f"{'batch':>6} {'1 collective':>14} {'2 collectives':>15}")
for batch in (1, 32, 256):
  one = comm_over_compute(batch, 4096, 8, BANDWIDTH_RATIO, 1)
  two = comm_over_compute(batch, 4096, 8, BANDWIDTH_RATIO, 2)
  print(f'{batch:>6} {one:>14.3f} {two:>15.3f}')
print('\n(the ratio of talking to computing. 1.0 means half your step is the network.)')
# The batch at which talking becomes longer than computing, for three machines.
for bandwidth_ratio, name in ((1.0, 'NVLink'), (25.0, 'PCIe'), (200.0, 'Ethernet')):
  one = break_even_batch(4096, 8, bandwidth_ratio, 1)
  two = break_even_batch(4096, 8, bandwidth_ratio, 2)
  print(f'{name:>9}: talking becomes longer than computing at batch {one:>7,.0f} '
        f'with 1 collective, {two:>7,.0f} with 2')

### Both arrangements are correct. Only one is usable.

Column-then-row needs one all-reduce for each block. Row-then-column needs
two, because `relu` is not linear. You cannot put a partial sum through it.

That is the whole design rule, and it is general: **put the collective where
the non-linearity is not**.

Attention splits the same way for the same reason. Each rank owns whole heads,
because softmax reduces over a row, and one rank must own the whole row.

### And the number that decides if any of this is worth it

At batch 1 the collectives are a rounding error, even on PCIe. At batch 256
with eight ranks they are most of the step. If you double them, the step
becomes communication.

So tensor parallelism and continuous batching pull against each other. Every
plot in Part 2 told you to raise the batch. This one says that the
interconnect also has a vote.

On NVLink the interconnect does not care. On PCIe it sets your maximum batch
size. That sets your throughput, and throughput was the thing that you split
the model to get.

Stage 20 builds this with real collectives. It checks that the two-rank output
matches the one-rank output exactly. The lesson is where the communication
lands, and not the speed. On one GPU there is no speed to get.

    ./vc guide 20